# Aletheia — Stage 1 exploratory analysis

This notebook reports the **exploratory** Stage 1 results of the Aletheia
plurality-audit pipeline. Per `PROTOCOL.md`, Stage 1 is not confirmatory:
it exists to measure effect sizes and calibrate thresholds for Stage 2.

The findings below are what motivated the Stage 2 protocol. They are
reported here so that the sequence *explore → calibrate → confirm* is
auditable.

**Nothing here is presented as a final result.** The confirmatory run
lives in `02_confirmatory.ipynb`.

## 1. Setup

We load the results of the Stage 1 sweep from `data/results/`. The sweep
ran Aletheia (with OpenAI `gpt-4o-mini` as the claim normalizer) and four
naive baselines on three problem difficulties:

| Config | `n_positions` | True inflation `I` |
|--------|---------------|---------------------|
| easy   | 10            | 3.0                 |
| medium | 5             | 6.0                 |
| hard   | 2             | 15.0                |

Each configuration used 30 threads with 30 voices and `medium` paraphrase
level. No noise was added.

In [ ]:
import json
from pathlib import Path

from aletheia.evaluation import ThreadResult
from aletheia.metrics import ClusteringMetrics
from aletheia.visualize import (
    figure_clustering_quality,
    figure_improvement_gauge,
    figure_inflation_scatter,
    figure_median_error_comparison,
    figure_per_thread_errors,
)

RESULTS_DIR = Path("..") / "data" / "results"
SWEEP_FILE = RESULTS_DIR / "stage1_sweep.json"

## 2. Loading the sweep

The sweep summary is a list of rows, one per configuration.

In [ ]:
sweep = json.loads(SWEEP_FILE.read_text(encoding="utf-8"))

print(f"{'n_pos':>6} {'I_true':>8} {'Aletheia err':>14} "
      f"{'Best base err':>15} {'Improv':>10} "
      f"{'Aletheia ARI':>14} {'Best base ARI':>15}")
print("-" * 96)
for row in sweep:
    print(
        f"{row['n_positions']:>6} "
        f"{row['true_inflation']:>8.2f} "
        f"{row['aletheia_err']:>14.4f} "
        f"{row['best_baseline_err']:>15.4f} "
        f"{row['improvement']:>9.2%} "
        f"{row['aletheia_ari']:>14.4f} "
        f"{row['best_baseline_ari']:>15.4f}"
    )

## 3. Interpretation

Two findings stand out:

1. **Aletheia wins where the task is to distinguish many positions.** For
   `n_positions ∈ {5, 10}` (inflation `I ∈ {3, 6}`), Aletheia reduces
   median relative error on `I` by 36–77% relative to the best baseline,
   and improves median ARI from ~0.51 to ~0.86.

2. **Aletheia loses where the task is to recognize paraphrases of few
   positions.** For `n_positions = 2` (inflation `I = 15`), Aletheia is
   worse than the best baseline by 20%. The failure mode is
   over-fragmentation: the LLM splits paraphrases of the same position
   into multiple clusters.

Finding 2 is documented as a scope limitation in `PROTOCOL.md`
(Amendment 2) and in the README. It is not hidden.

## 4. Figures

The figures below are the same ones exported to `docs/figures/` for the
site. They are generated by functions in `src/aletheia/visualize.py`, so
the notebook, the Streamlit app, and the static site show identical
content.

### 4.1 Median relative error across difficulties

In [ ]:
def _clustering_metrics_from_dict(d: dict) -> ClusteringMetrics:
    from dataclasses import fields
    allowed = {f.name for f in fields(ClusteringMetrics)}
    return ClusteringMetrics(**{k: v for k, v in d.items() if k in allowed})


def _thread_results_from_json(path: Path) -> list[ThreadResult]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    results = []
    for entry in payload["per_thread"]:
        baselines = {
            name: _clustering_metrics_from_dict(m)
            for name, m in entry["baselines"].items()
        }
        results.append(
            ThreadResult(
                seed=entry["seed"],
                n_voices=entry["n_voices"],
                n_positions=entry["n_positions"],
                true_inflation=entry["true_inflation"],
                aletheia=_clustering_metrics_from_dict(entry["aletheia"]),
                baselines=baselines,
            )
        )
    return results

In [ ]:
for n_pos in (2, 5, 10):
    path = RESULTS_DIR / f"stage1_npos{n_pos}.json"
    results = _thread_results_from_json(path)
    figure_median_error_comparison(results).show()

### 4.2 Clustering quality (ARI and NMI)

In [ ]:
for n_pos in (2, 5, 10):
    path = RESULTS_DIR / f"stage1_npos{n_pos}.json"
    results = _thread_results_from_json(path)
    figure_clustering_quality(results).show()

### 4.3 Per-thread error distribution

In [ ]:
for n_pos in (2, 5, 10):
    path = RESULTS_DIR / f"stage1_npos{n_pos}.json"
    results = _thread_results_from_json(path)
    figure_per_thread_errors(results).show()

### 4.4 Predicted vs true inflation

In [ ]:
for n_pos in (2, 5, 10):
    path = RESULTS_DIR / f"stage1_npos{n_pos}.json"
    results = _thread_results_from_json(path)
    figure_inflation_scatter(results).show()

### 4.5 Improvement over best baseline

In [ ]:
for n_pos in (2, 5, 10):
    path = RESULTS_DIR / f"stage1_npos{n_pos}.json"
    results = _thread_results_from_json(path)
    figure_improvement_gauge(results).show()

## 5. What Stage 2 must test

Based on these exploratory findings, the Stage 2 protocol tests the
following **frozen** hypotheses (see `PROTOCOL.md`, Amendments 2 and 3):

- **H1 (conditional):** Aletheia beats the best baseline by **≥ 15%**
  relative on `I ∈ {3, 6}` (calibrated at 50% of the observed effect).
- **H1 (negative control):** Aletheia is expected **not** to beat the
  best baseline on `I = 15`, due to the over-fragmentation documented in
  Amendment 3.
- **H2 (ablation):** LLM normalization reduces error by **≥ 20%**
  relative to raw embeddings.

Stage 2 will run on fresh seeds and will report both positive and
negative outcomes with equal prominence.